In [ ]:
from pathlib import Path
import logging

import h5py
import numpy as np
import pandas as pd


INPUT_DIR = Path(r"C:\Users\topohl\Documents\SLEAP\Projects\AnalysisNOR\predictions\geom")
OUTPUT_DIR = INPUT_DIR

GEOM_NAMES = ["tl", "tr", "br", "bl"]
OVERWRITE = True

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")


def load_sleap_tracks(path: Path) -> np.ndarray:
    """Load SLEAP tracks as frames x geom_points x coords x instances."""
    with h5py.File(path, "r") as f:
        if "tracks" not in f:
            raise KeyError(f"No 'tracks' dataset found in {path.name}")

        raw = f["tracks"][:]

    if raw.ndim != 4:
        raise ValueError(f"{path.name}: expected 4D tracks array, got shape {raw.shape}")

    # SLEAP usual format: nodes x coords x frames x instances
    tracks = np.transpose(raw, (2, 0, 1, 3))

    n_frames, n_nodes, n_coords, n_instances = tracks.shape

    if n_nodes != len(GEOM_NAMES):
        raise ValueError(
            f"{path.name}: expected {len(GEOM_NAMES)} geometry points, "
            f"found {n_nodes}. Shape after transpose: {tracks.shape}"
        )

    if n_coords < 2:
        raise ValueError(f"{path.name}: expected at least x/y coordinates, found {n_coords}")

    return tracks[:, :, :2, :]


def estimate_static_geometry(tracks: np.ndarray) -> tuple[np.ndarray, pd.DataFrame]:
    """Estimate one static x/y coordinate per geometry point using the median."""
    n_frames, n_points, _, n_instances = tracks.shape

    static_geom = np.full((n_points, 2), np.nan)
    qc_rows = []

    for point_idx, point_name in enumerate(GEOM_NAMES):
        for coord_idx, coord_name in enumerate(["x", "y"]):
            values = tracks[:, point_idx, coord_idx, :].reshape(-1)
            valid = ~np.isnan(values)

            n_valid = int(valid.sum())
            missing_fraction = float(np.isnan(values).mean())

            if n_valid > 0:
                static_geom[point_idx, coord_idx] = np.nanmedian(values)

            qc_rows.append({
                "point": point_name,
                "coord": coord_name,
                "n_frames": n_frames,
                "n_instances": n_instances,
                "n_values": int(values.size),
                "n_valid": n_valid,
                "missing_fraction": missing_fraction,
                "median_value": static_geom[point_idx, coord_idx],
            })

    qc = pd.DataFrame(qc_rows)
    return static_geom, qc


def static_geometry_to_dataframe(static_geom: np.ndarray, n_frames: int) -> pd.DataFrame:
    """Create one output row per frame with constant static geometry coordinates."""
    data = {"frame": np.arange(n_frames)}

    for point_idx, point_name in enumerate(GEOM_NAMES):
        data[f"{point_name}_x"] = np.repeat(static_geom[point_idx, 0], n_frames)
        data[f"{point_name}_y"] = np.repeat(static_geom[point_idx, 1], n_frames)

    return pd.DataFrame(data)


def process_file(input_path: Path) -> None:
    output_path = OUTPUT_DIR / f"{input_path.stem}_locs.csv"
    qc_path = OUTPUT_DIR / f"{input_path.stem}_geom_qc.csv"

    if output_path.exists() and not OVERWRITE:
        logging.info("Skipping existing output: %s", output_path.name)
        return

    tracks = load_sleap_tracks(input_path)

    static_geom, qc = estimate_static_geometry(tracks)
    df = static_geometry_to_dataframe(static_geom, n_frames=tracks.shape[0])

    df.to_csv(output_path, index=False)
    qc.to_csv(qc_path, index=False)

    logging.info(
        "Processed %s -> %s | frames=%d",
        input_path.name,
        output_path.name,
        len(df),
    )


def main() -> None:
    h5_files = sorted(INPUT_DIR.glob("*.h5"))

    if not h5_files:
        raise FileNotFoundError(f"No .h5 files found in {INPUT_DIR}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for input_path in h5_files:
        try:
            process_file(input_path)
        except Exception as exc:
            logging.error("Failed processing %s: %s", input_path.name, exc)


if __name__ == "__main__":
    main()